# NeuroLens — Precompute Cache

Run this notebook **once** on a GPU runtime to generate the NeuroLens cache.
This processes the stimulus library through TRIBE v2 and comparison models.

**Requirements:** Colab GPU runtime, HuggingFace account (for LLaMA access)

In [ ]:
import sys, os
from pathlib import Path

# Clone repo and set up environment (Colab)
if not os.path.exists('neurolens'):
    if not os.path.exists('tribev2'):
        !git clone https://github.com/Ansumanbhujabal/tribev2.git
    os.chdir('tribev2')

sys.path.insert(0, os.getcwd())

# Install dependencies (ignore numpy version conflict — it works fine)
!pip install -q -e '.[plotting]' --no-deps 2>/dev/null || true
!pip install -q exca neuralset neuraltrain einops pyyaml moviepy huggingface_hub \
    gtts langdetect spacy soundfile Levenshtein julius transformers x_transformers \
    nilearn matplotlib scipy pydantic tqdm open_clip_torch 2>/dev/null

import json
import numpy as np
import torch
from tribev2 import TribeModel

CACHE_DIR = Path('neurolens_cache')
CACHE_DIR.mkdir(exist_ok=True)
for subdir in ['stimuli', 'brain_preds', 'roi_summaries', 'embeddings']:
    (CACHE_DIR / subdir).mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Stimulus library — add your video/audio files to neurolens/videos/
STIMULI = [
    {'id': 'clip_001', 'name': 'Jack Ma Motivation', 'category': 'Speech',
     'media_type': 'video', 'duration_sec': 10.0, 'path': 'neurolens/videos/Jack_ma_motivation.mp4'},
    {'id': 'clip_002', 'name': 'Muay Thai Kick', 'category': 'Silence + Visuals',
     'media_type': 'video', 'duration_sec': 8.0, 'path': 'neurolens/videos/Muay_thai_kick.mp4'},
    # Add more stimuli here...
]

# Save metadata (without paths — those are only needed during precompute)
metadata = [{k: v for k, v in s.items() if k != 'path'} for s in STIMULI]
(CACHE_DIR / 'stimuli' / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Saved metadata for {len(STIMULI)} stimuli')

In [ ]:
# Load TRIBE v2
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")

from neurolens.roi import summarize_by_roi_group

for stim in STIMULI:
    sid = stim["id"]
    print(f"Processing {sid}: {stim['name']}...")

    # Get events and predict
    media_kwarg = {f"{stim['media_type']}_path": stim["path"]}
    events = model.get_events_dataframe(**media_kwarg)
    preds, segments = model.predict(events)

    # Save brain predictions
    np.savez(CACHE_DIR / "brain_preds" / f"{sid}.npz", preds=preds)

    # Save ROI summary (average across timesteps)
    avg_pred = preds.mean(axis=0)
    roi_summary = summarize_by_roi_group(avg_pred)
    (CACHE_DIR / "roi_summaries" / f"{sid}.json").write_text(json.dumps(roi_summary))

    print(f"  -> {preds.shape[0]} timesteps, {preds.shape[1]} vertices")

In [ ]:
import open_clip

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

(CACHE_DIR / "embeddings" / "clip").mkdir(exist_ok=True)

from torchvision.io import read_video

for stim in STIMULI:
    sid = stim["id"]
    if stim["media_type"] != "video":
        continue
    # Read middle frame
    video, _, _ = read_video(stim["path"], pts_unit="sec")
    mid_frame = video[len(video) // 2]  # (H, W, C)
    img = preprocess(mid_frame.permute(2, 0, 1)).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(img).squeeze().cpu()
    torch.save(emb, CACHE_DIR / "embeddings" / "clip" / f"{sid}.pt")
    print(f"CLIP embedding for {sid}: {emb.shape}")

In [ ]:
from transformers import WhisperModel, WhisperFeatureExtractor
import soundfile as sf

whisper = WhisperModel.from_pretrained("openai/whisper-base").to(device).eval()
whisper_fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")

(CACHE_DIR / "embeddings" / "whisper").mkdir(exist_ok=True)

for stim in STIMULI:
    sid = stim["id"]
    audio_path = stim["path"].replace(".mp4", ".wav")
    try:
        audio, sr = sf.read(audio_path)
        inputs = whisper_fe(audio, sampling_rate=sr, return_tensors="pt").to(device)
        with torch.no_grad():
            out = whisper.encoder(**inputs)
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu()
        torch.save(emb, CACHE_DIR / "embeddings" / "whisper" / f"{sid}.pt")
        print(f"Whisper embedding for {sid}: {emb.shape}")
    except Exception as e:
        print(f"Skipped Whisper for {sid}: {e}")

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer

gpt2 = GPT2Model.from_pretrained("gpt2").to(device).eval()
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")

(CACHE_DIR / "embeddings" / "gpt2").mkdir(exist_ok=True)

for stim in STIMULI:
    sid = stim["id"]
    text = f"{stim['name']}. Category: {stim['category']}"
    inputs = gpt2_tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gpt2(**inputs)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu()
    torch.save(emb, CACHE_DIR / "embeddings" / "gpt2" / f"{sid}.pt")
    print(f"GPT-2 embedding for {sid}: {emb.shape}")

## Done!

Cache generated at `neurolens_cache/`. Download this folder or upload it to
Google Drive / HuggingFace Hub for use with the main `neurolens.ipynb` notebook.